<a href="https://colab.research.google.com/github/rozmar/Analyzing-Open-Neuroscience-Data-SZTE-2027/blob/main/Modul%202/00_Build_Course_Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 0: Build the course data

This notebook downloads two Allen Institute Patch-seq datasets from their **original public sources**
and turns them into a handful of small, clean tables that the rest of Module 2 loads in seconds:

* **Mouse** visual cortex GABAergic interneurons: Gouwens, Sorensen et al. (2020) *Cell* 183:935,
  [doi:10.1016/j.cell.2020.09.057](https://doi.org/10.1016/j.cell.2020.09.057)
* **Human** neocortex GABAergic interneurons: Lee, Dalley et al. (2023) *Science* 382:eadf6484,
  [doi:10.1126/science.adf6484](https://doi.org/10.1126/science.adf6484)

Nothing here is hidden: every input is listed in `SOURCES` below with its URL, every download is
cached and fingerprinted (sha256), and every transformation is a few lines of plain pandas/numpy
that you can read top to bottom.

**How to use it:** run all cells (*Runtime > Run all* on Colab). It takes about a minute, downloads
~120 MB and needs ~2 GB of RAM. The results are written to the `data/` folder:

| file | content |
|---|---|
| `cells.csv` | one row per cell, both species, all identifiers & labels |
| `mouse_ephys_features.csv`, `human_ephys_features.csv` | precomputed electrophysiology features (Allen) |
| `mouse_morph_features.csv`, `human_morph_features.csv` | precomputed morphology features (Allen) |
| `mouse_expression_selected.parquet`, `human_expression_selected.parquet` | log2(CPM+1) for a curated gene set (pandas-only access) |
| `mouse_expression.h5ad` | all genes, sparse (AnnData); `X` = raw counts |
| `human_expression.h5ad` | all genes, sparse (AnnData); log2(CPM+1) as published |
| `genes_selected.csv` | which genes are in the `*_selected` files and why |
| `DATA_DICTIONARY.md` | every column of every table, incl. original column names |
| `provenance.json` | sources, checksums, versions, dates |
| `validation_report.txt` | counts and consistency checks: **read it after every run** |

The one file you do not download here is `data/SpecimenMetadata.csv` (the Brain Knowledge Platform
specimen table). It has no stable download link, so it is part of the course repository; see
`data/README.md` for how it was exported.

## 1. Setup

On **Google Colab** this cell clones the course repository (for `data/SpecimenMetadata.csv`),
moves into `Modul 2/` and installs the two packages Colab does not have. Files on Colab disappear
when the runtime is reset; just run this notebook again.

**Locally** nothing is installed here: run `uv sync` once in the repository root and
choose the `.venv` kernel.

In [ ]:
import os
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/Analyzing-Open-Neuroscience-Data-SZTE-2027"
    if not os.path.exists(REPO):
        !git clone --depth 1 https://github.com/rozmar/Analyzing-Open-Neuroscience-Data-SZTE-2027.git {REPO}
    os.chdir(REPO + "/Modul 2")
    %pip install -q anndata rdata
print("working directory:", os.getcwd())

## 2. Settings

The defaults build everything. Change them only if you need to, e.g. `SKIP_EXPRESSION = True` for a
quick run without the (large) expression matrices, or `OFFLINE = True` to use only files that are
already in `data/downloads/`.

In [ ]:
from pathlib import Path
from types import SimpleNamespace

DATA_DIR = Path("data").resolve()        # outputs, downloads and SpecimenMetadata.csv live here

ONLY = None                              # None = both species, or "mouse" / "human"
SKIP_EXPRESSION = False                  # True = skip the (large) expression matrices
SKIP_DANDI = False                       # True = do not query the DANDI API for NWB files
OFFLINE = False                          # True = use cached downloads only
MOUSE_SPECIMENS = DATA_DIR / "SpecimenMetadata.csv"   # Brain Knowledge Platform specimen export
MOUSE_COUNTS = "allen-parquet"           # or "nemo" = the original (large) NeMO CSV

args = SimpleNamespace(out=DATA_DIR, cache=DATA_DIR / "downloads", only=ONLY,
                       skip_expression=SKIP_EXPRESSION, skip_dandi=SKIP_DANDI, offline=OFFLINE,
                       mouse_specimens=MOUSE_SPECIMENS, mouse_counts=MOUSE_COUNTS)

In [ ]:
import datetime as dt
import hashlib
import json
import platform
import re
import tarfile
import time
from importlib import metadata as importlib_metadata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import scipy.sparse as sp

SCRIPT_VERSION = "0.1.0"

## 3. Sources

Every file the notebook reads, and where it comes from. `"urls"` are tried in order; the one that
worked is recorded in `provenance.json`.

In [ ]:
GH_HUMAN = ("https://raw.githubusercontent.com/AllenInstitute/human_patchseq_gaba/"
            "5eb357a1cea246c7c847b5608a8fc7e87ea95b58/data/")      # pinned commit (2023-11-28)
ALLEN_CDN = "https://cdn.prod.website-files.com/689cfbd308fa7373b604d290/"

SOURCES = {
    # ---------------- mouse (Gouwens et al. 2020) ----------------
    "mouse_metadata": {
        "file": "20200711_patchseq_metadata_mouse.csv",
        "urls": [ALLEN_CDN + "68d61b068b5cfcfd77c92050_20200711_patchseq_metadata_mouse.csv"],
        "about": "Allen cell metadata (IDs, donor, genotype, area/layer, t-type alias). "
                 "Linked as 'Download cell metadata' on brain-map.org > Multimodal "
                 "Characterization of Individual Neurons.",
    },
    "mouse_file_manifest": {
        "file": "2021-09-13_mouse_file_manifest.xlsx",
        "urls": [ALLEN_CDN + "68d61b4a9560fc6f223d80f1_2021-09-13_mouse_file_manifest.xlsx"],
        "about": "Allen per-cell raw file manifest: DANDI NWB asset URL and Brain Image Library "
                 "SWC URL for each cell, with checksums.",
    },
    "mouse_specimens": {
        "file": "SpecimenMetadata.csv",
        "urls": [],                                       # never downloaded: must be exported by hand
        "about": "Specimen table of the Brain Knowledge Platform project 1HEYEW7GMUKWIQW37BO "
                 "(t-type, subclass, MET-type, precomputed ephys + morphology features). It has no "
                 "stable download URL: it is exported by hand from "
                 "https://knowledge.brain-map.org/data/1HEYEW7GMUKWIQW37BO/specimens "
                 "(gear icon > show all columns > Download Data > Metadata table). "
                 "Expected as data/SpecimenMetadata.csv; another location via MOUSE_SPECIMENS.",
    },
    "mouse_counts_allen_parquet": {
        "file": "patchseq_count_matrix.parquet",
        "urls": ["https://github.com/AllenInstitute/EducationAndEngagement/releases/download/"
                 "patchseq-data-v1/patchseq_count_matrix.parquet"],
        "about": "Allen Institute (Education & Engagement) parquet conversion of the NeMO count "
                 "matrix 20200513_Mouse_PatchSeq_Release_count.v2 (45,768 genes x 4,435 cells).",
    },
    "mouse_counts_nemo": {
        "file": "20200513_Mouse_PatchSeq_Release_count.v2.csv.tar",
        "urls": ["https://data.nemoarchive.org/other/AIBS/AIBS_patchseq/transcriptome/scell/SMARTseq/"
                 "processed/analysis/20200611/20200513_Mouse_PatchSeq_Release_count.v2.csv.tar"],
        "about": "Original count matrix on the Neuroscience Multi-omic Archive (NeMO). Large CSV.",
    },
    # ---------------- human (Lee, Dalley et al. 2023) ----------------
    "human_metadata": {
        "file": "LeeDalley_manuscript_metadata_v2.csv",
        "urls": [GH_HUMAN + "LeeDalley_manuscript_metadata_v2.csv"],
        "about": "Cell metadata of Lee, Dalley et al. 2023 (AllenInstitute/human_patchseq_gaba).",
    },
    "human_ephys": {
        "file": "LeeDalley_ephys_fx.csv",
        "urls": [GH_HUMAN + "LeeDalley_ephys_fx.csv"],
        "about": "Precomputed electrophysiology features (IPFX naming).",
    },
    "human_morph": {
        "file": "LeeDalley_morpho_features.csv",
        "urls": [GH_HUMAN + "LeeDalley_morpho_features.csv"],
        "about": "Precomputed morphology features.",
    },
    "human_expr_1": {
        "file": "complete_patchseq_data_sets1.RData",
        "urls": [GH_HUMAN + "complete_patchseq_data_sets1.RData"],
        "about": "Gene expression, part 1 (R object datPatch1: genes x cells, log2(CPM+1)).",
    },
    "human_expr_2": {
        "file": "complete_patchseq_data_sets2.RData",
        "urls": [GH_HUMAN + "complete_patchseq_data_sets2.RData"],
        "about": "Gene expression, part 2 (datPatch2) + annotation tables (annoPatch, metaPatch).",
    },
    "vgic_genes": {
        "file": "vgic_genes.csv",
        "urls": [GH_HUMAN + "vgic_genes.csv"],
        "about": "Allen-curated list of voltage-gated ion channel genes (HGNC symbols).",
    },
}

DANDI_API = "https://api.dandiarchive.org/api"
DANDISETS = {
    "mouse": {"id": "000020", "version": "0.210913.1639"},   # version linked by the Allen download page
    "human": {"id": "000636", "version": None},              # None = most recent published version
}

Hand-picked marker genes, and the coarse mouse/human subclass mapping used for cross-species
comparisons.

In [ ]:
# Hand-curated marker genes (mouse symbols). Human symbols = upper case.
MARKER_GENES = [
    # pan-neuronal / class
    "Snap25", "Gad1", "Gad2", "Slc32a1", "Slc17a7",
    # GABAergic subclasses and types
    "Pvalb", "Sst", "Vip", "Lamp5", "Sncg", "Serpinf1", "Chodl", "Nos1", "Lhx6", "Nkx2-1",
    "Adarb2", "Prox1", "Pax6", "Htr3a", "Ndnf", "Reln", "Calb1", "Calb2", "Cck", "Npy",
    "Tac1", "Tac2", "Crh", "Cort", "Pthlh", "Th", "Chat", "Cnr1", "Kit", "Nr2f2", "Cxcl14",
    # layer markers of glutamatergic neurons (contamination / comparison)
    "Cux2", "Rorb", "Fezf2", "Foxp2",
]

# Marker genes whose human ortholog is not simply the upper-case symbol.
MOUSE_TO_HUMAN_EXCEPTIONS = {"Tac2": "TAC3"}   # mouse Tac2 (neurokinin B) = human TAC3

# Coarse cross-species subclass harmonization (documented approximation, see DATA_DICTIONARY).
SUBCLASS_HARMONIZED = {
    "Pvalb": "Pvalb", "Sst": "Sst", "Sst Chodl": "Sst", "Vip": "Vip",
    "Lamp5": "Lamp5/Pax6/Other", "Sncg": "Lamp5/Pax6/Other", "Serpinf1": "Lamp5/Pax6/Other",
    "PVALB": "Pvalb", "SST": "Sst", "VIP": "Vip", "LAMP5/PAX6/Other": "Lamp5/Pax6/Other",
}

## 4. Helpers

Logging (every `check` line also goes into `validation_report.txt`), cached downloads, column renaming.

In [ ]:
class Log:
    lines: list[str] = []

    @classmethod
    def info(cls, msg: str) -> None:
        print(msg, flush=True)

    @classmethod
    def check(cls, msg: str) -> None:
        """A line that also goes into validation_report.txt."""
        cls.lines.append(msg)
        print("  [check] " + msg, flush=True)

    @classmethod
    def warn(cls, msg: str) -> None:
        cls.lines.append("WARNING: " + msg)
        print("  [WARNING] " + msg, flush=True)


PROVENANCE: dict = {"sources": {}, "dandi": {}}


def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def fetch(key: str, cache: Path, offline: bool = False, override: Path | None = None) -> Path | None:
    """Return a local path for SOURCES[key], downloading it into the cache if needed."""
    src = SOURCES[key]
    if override:
        path = Path(override)
        PROVENANCE["sources"][key] = {"local_file": str(path), "sha256": sha256(path),
                                      "size_bytes": path.stat().st_size, "about": src["about"],
                                      "note": "manual export from the Brain Knowledge Platform"}
        return path
    path = cache / src["file"]
    sidecar = path.with_name(path.name + ".source.json")   # remembers where a cached file came from
    used_url = None
    if path.exists():
        used_url = json.loads(sidecar.read_text())["url"] if sidecar.exists() else "unknown (cached)"
    elif offline:
        Log.warn(f"{key}: not in cache and OFFLINE = True")
        return None
    else:
        for url in src["urls"]:
            try:
                Log.info(f"  downloading {key}: {url}")
                with requests.get(url, stream=True, timeout=60) as r:
                    r.raise_for_status()
                    tmp = path.with_suffix(path.suffix + ".part")
                    with open(tmp, "wb") as f:
                        for chunk in r.iter_content(1 << 20):
                            f.write(chunk)
                    tmp.rename(path)
                used_url = url
                sidecar.write_text(json.dumps({"url": url, "downloaded": dt.datetime.now().isoformat()}))
                if url != src["urls"][0]:
                    Log.warn(f"{key}: primary URL failed, used fallback {url}")
                break
            except Exception as e:  # noqa: BLE001 - report and try the next URL
                Log.info(f"    failed ({type(e).__name__}: {e})")
        if used_url is None:
            Log.warn(f"{key}: could not be downloaded from any URL")
            return None
    PROVENANCE["sources"][key] = {
        "url": used_url, "file": src["file"], "sha256": sha256(path),
        "size_bytes": path.stat().st_size, "about": src["about"],
        "retrieved": dt.datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
    }
    return path


def snake(name: str) -> str:
    """'Threshold V (long square) (millivolts)' -> 'threshold_v_long_square_mv'."""
    units = {"(millivolts)": "_mv", "(picoamps)": "_pa", "(millimeters)": "_mm"}
    suffix = ""
    for u, s in units.items():
        if u in name:
            name, suffix = name.replace(u, ""), s
    name = name.replace("/", "_").replace("%", "pct")
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name).strip("_").lower()
    return name + suffix


def to_int_or_na(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")


def first_column(df: pd.DataFrame, *names: str) -> pd.Series:
    """The first of `names` that exists in df (column names change between releases); all-NA if none."""
    for name in names:
        if name in df:
            return df[name]
    Log.warn(f"none of the columns {names} found; filled with NA")
    return pd.Series(pd.NA, index=df.index, dtype="object")


HUMAN_TO_MOUSE_EXCEPTIONS = {"SCN2A": "Scn2a1"}   # the 2020 mouse matrix uses the older symbol


def human_symbol_to_mouse(sym: str) -> str:
    """Orthology by capitalization (KCNC2 -> Kcnc2). Works for most ion channel genes;
    genes that do not exist in the mouse matrix are reported, not silently dropped."""
    return HUMAN_TO_MOUSE_EXCEPTIONS.get(sym, sym[:1].upper() + sym[1:].lower())

## 5. DANDI: find the NWB file of every cell

The raw electrophysiology recordings are on the [DANDI archive](https://dandiarchive.org). The session id in each file name links it to a cell.

In [ ]:
def dandi_assets(dandiset: str, version: str | None) -> tuple[pd.DataFrame, str]:
    if version is None:
        info = requests.get(f"{DANDI_API}/dandisets/{dandiset}/", timeout=60).json()
        version = (info.get("most_recent_published_version") or {}).get("version") or "draft"
    url = f"{DANDI_API}/dandisets/{dandiset}/versions/{version}/assets/?page_size=1000"
    rows = []
    while url:
        page = requests.get(url, timeout=120).json()
        rows += [{"dandi_path": a["path"], "dandi_asset_id": a["asset_id"], "size_bytes": a["size"]}
                 for a in page["results"]]
        url = page.get("next")
        time.sleep(0.2)
    df = pd.DataFrame(rows)
    df["ephys_session_id"] = to_int_or_na(df["dandi_path"].str.extract(r"ses-(\d+)")[0])
    return df, version


def attach_dandi(cells: pd.DataFrame, species: str, skip: bool) -> pd.DataFrame:
    ds = DANDISETS[species]
    cells["dandiset"] = "DANDI:" + ds["id"]
    if skip:
        Log.warn(f"{species}: DANDI lookup skipped (SKIP_DANDI = True)")
        return cells
    try:
        assets, version = dandi_assets(ds["id"], ds["version"])
    except Exception as e:  # noqa: BLE001
        Log.warn(f"{species}: DANDI API not reachable ({type(e).__name__}); dandi_path left empty")
        return cells
    PROVENANCE["dandi"][species] = {"dandiset": ds["id"], "version": version, "n_assets": len(assets)}
    assets = assets.dropna(subset=["ephys_session_id"]).drop_duplicates("ephys_session_id")
    out = cells.drop(columns=[c for c in ["dandi_path", "dandi_asset_id"] if c in cells]).merge(
        assets[["ephys_session_id", "dandi_path", "dandi_asset_id"]], on="ephys_session_id", how="left")
    out["dandi_version"] = version
    filled = out["dandi_asset_id"].notna()
    out.loc[filled, "nwb_url"] = DANDI_API + "/assets/" + out.loc[filled, "dandi_asset_id"] + "/download/"
    n_e = int(out["has_ephys"].sum())
    n_m = int((out["has_ephys"] & filled).sum())
    Log.check(f"{species}: DANDI {ds['id']} v{version}: {len(assets)} NWB assets; "
              f"{n_m}/{n_e} cells with ephys matched to an NWB file by session id")
    return out

## 6. Mouse (Gouwens et al. 2020)

Combines the Allen cell metadata, the specimen table (t-types, MET-types, precomputed features) and
the raw-file manifest into one row per cell, then loads the count matrix.

In [ ]:
def build_mouse(args) -> dict:
    Log.info("\n=== MOUSE (Gouwens et al. 2020) ===")
    cache = args.cache
    meta = pd.read_csv(fetch("mouse_metadata", cache, args.offline))
    spec_path = Path(args.mouse_specimens)
    if not spec_path.exists():
        raise FileNotFoundError(f"mouse specimen table not found: {spec_path}\n"
                                "Export it from the Brain Knowledge Platform (see data/README.md) and save it as "
                                f"data/{SOURCES['mouse_specimens']['file']}, or set MOUSE_SPECIMENS.")
    spec = pd.read_csv(fetch("mouse_specimens", cache, override=spec_path))
    man_path = fetch("mouse_file_manifest", cache, args.offline)
    manifest = pd.read_excel(man_path) if man_path else None

    Log.check(f"mouse metadata: {len(meta)} cells, {meta['cell_specimen_id'].duplicated().sum()} duplicate ids")

    # --- cells table ---------------------------------------------------------
    c = pd.DataFrame({
        "cell_id": "mouse_" + meta["cell_specimen_id"].astype(str),
        "species": "mouse",
        "dataset": "Gouwens2020",
        "specimen_id": meta["cell_specimen_id"].astype("Int64"),
        "donor_id": meta["donor_id"].astype(str),
        "sex": meta["biological_sex"],
        "age_days": to_int_or_na(meta["age"].astype(str).str.lstrip("P")),
        "genotype": meta["full_genotype"],
        "hemisphere": meta["hemisphere"],
        "structure": meta["structure"],
        "area": meta["structure"].str.extract(r"^([A-Za-z]+?)(?:\d|$)")[0],
        "layer": meta["structure"].str.extract(r"(\d.*)$")[0],
        "soma_depth_normalized": meta["cell_soma_normalized_depth"],
        "dendrite_type": meta["dendrite_type"],
        "reconstruction_type": meta["neuron_reconstruction_type"],
        "preparation": "acute",
        # renamed in the 2025 re-upload of the metadata file
        "t_type": first_column(meta, "T-type Label", "corresponding_AIT2.3.1_alias"),
        "met_type": first_column(meta, "MET-type Label"),
        "transcriptomic_sample_id": meta["transcriptomics_sample_id"],
        "ephys_session_id": to_int_or_na(meta["ephys_session_id"]),
    })
    c["subclass"] = c["t_type"].str.split().str[0]
    c["in_published_analysis"] = False

    if spec is not None:
        spec = spec.drop_duplicates("Specimen ID")
        s = pd.DataFrame({
            "specimen_id": spec["Specimen ID"].astype("Int64"),
            "t_type_bkp": spec["T type"],
            "t_type_probability": spec["T type assignment probability"],
            "subclass_bkp": spec["T type sub-class"].str.replace(" interneuron", "", regex=False),
            "class": spec["T type class"].str.replace(" interneuron", "", regex=False),
            "met_type_bkp": spec["MET type"],
            "layer_bkp": spec["Cortical layer"].astype(str),
            "has_ephys_bkp": spec["Electrophysiology available"].astype(bool),
            "has_morphology_bkp": spec["Morphology available"].astype(bool),
        })
        c = c.merge(s, on="specimen_id", how="left")
        inpub = c["t_type_bkp"].notna()
        Log.check(f"mouse specimens table: {len(spec)} cells; {inpub.sum()} matched to metadata; "
                  f"{(~inpub).sum()} metadata cells are not in the published analysis")
        agree = (c.loc[inpub, "t_type"] == c.loc[inpub, "t_type_bkp"]).mean()
        Log.check(f"mouse t-type agreement metadata vs specimens table: {agree:.1%} (specimens table wins)")
        c.loc[inpub, "t_type"] = c.loc[inpub, "t_type_bkp"]
        c.loc[inpub, "subclass"] = c.loc[inpub, "subclass_bkp"]
        c.loc[inpub, "layer"] = c.loc[inpub, "layer_bkp"]
        c["met_type"] = c["met_type_bkp"].combine_first(c["met_type"])
        c["in_published_analysis"] = inpub
        c = c.drop(columns=["t_type_bkp", "subclass_bkp", "layer_bkp", "met_type_bkp"])
    else:
        Log.warn("mouse specimens table missing: MET-types from metadata only, no precomputed features")

    # --- raw file links from the Allen manifest --------------------------------
    c["nwb_url"] = pd.NA
    c["swc_url"] = pd.NA
    if manifest is not None:
        manifest["specimen_id"] = to_int_or_na(manifest["cell_specimen_id"])
        nwb = manifest[manifest["file_type"] == "nwb"].drop_duplicates("specimen_id")
        swc = manifest[manifest["file_type"] == "transformed_swc"].drop_duplicates("specimen_id")
        nwb = nwb.set_index("specimen_id")["archive_uri"]
        swc = swc.set_index("specimen_id")["archive_uri"].str.replace(
            "ftp://download.brainlib.org:8811/", "https://download.brainimagelibrary.org/", regex=False)
        c["nwb_url"] = c["specimen_id"].map(nwb)
        c["swc_url"] = c["specimen_id"].map(swc)
        c["dandi_asset_id"] = c["nwb_url"].str.extract(r"/assets/([0-9a-f-]{36})/")[0]
        Log.check(f"mouse manifest: {c['nwb_url'].notna().sum()} cells with NWB, "
                  f"{c['swc_url'].notna().sum()} with SWC (file types: "
                  f"{sorted(manifest['file_type'].dropna().unique().tolist())})")
    else:
        Log.warn("mouse file manifest missing: no NWB/SWC links from Allen")

    c["has_ephys"] = c.get("has_ephys_bkp", pd.Series(False, index=c.index)).eq(True) | c["nwb_url"].notna()
    c["has_morphology"] = c.get("has_morphology_bkp", pd.Series(False, index=c.index)).eq(True) \
        | c["swc_url"].notna()
    c = c.drop(columns=[x for x in ["has_ephys_bkp", "has_morphology_bkp"] if x in c])

    # --- feature tables ---------------------------------------------------------
    out = {}
    if spec is not None:
        cols = list(spec.columns)
        # select by name: column order differs between exports of the specimen table
        ephys_cols = [x for x in cols if re.match(r"(Threshold|Trough|Fast trough|Peak|Upstroke/downstroke|Vrest"
                                                   r"|sag|tau|Latency|Avg ISI|Adaptation|F I curve)", x)]
        morph_cols = [x for x in cols if x.startswith(("Axon ", "Basal Dendrite "))]
        for kind, sel, flag in [("ephys", ephys_cols, "Electrophysiology available"),
                                ("morph", morph_cols, "Morphology available")]:
            t = spec.loc[spec[flag].astype(bool), ["Specimen ID"] + sel].copy()
            rename = {o: snake(o) for o in sel}
            t = t.rename(columns=rename)
            t.insert(0, "cell_id", "mouse_" + t["Specimen ID"].astype(str))
            t = t.rename(columns={"Specimen ID": "specimen_id"})
            out[f"mouse_{kind}_features"] = (t, rename, "mouse_specimens")
            Log.check(f"mouse {kind} features: {len(t)} cells x {len(sel)} features")

    # --- expression -------------------------------------------------------------
    if not args.skip_expression:
        out["mouse_expression"] = mouse_expression(args, c)
        c["has_transcriptome"] = c["cell_id"].isin(out["mouse_expression"].obs_names)
    else:
        c["has_transcriptome"] = c["transcriptomic_sample_id"].notna()

    c = attach_dandi(c, "mouse", args.skip_dandi)
    out["cells"] = c
    return out


def mouse_expression(args, cells: pd.DataFrame):
    import anndata as ad
    if args.mouse_counts == "nemo":
        path = fetch("mouse_counts_nemo", args.cache, args.offline)
        with tarfile.open(path) as tar:
            member = next(m for m in tar.getmembers() if m.name.endswith(".csv"))
            chunks, genes = [], []
            for ch in pd.read_csv(tar.extractfile(member), index_col=0, chunksize=2000):
                genes += ch.index.astype(str).tolist()
                chunks.append(sp.csr_matrix(ch.to_numpy(dtype=np.int32)))
                samples = ch.columns.astype(str)
        X = sp.vstack(chunks).T.tocsr()                     # cells x genes
    else:
        import pyarrow.parquet as pq
        path = fetch("mouse_counts_allen_parquet", args.cache, args.offline)
        pf = pq.ParquetFile(path)
        genes = pf.read(columns=["gene_name"]).column(0).to_pylist()
        samples = [s for s in pf.schema_arrow.names if s != "gene_name"]
        blocks = []
        for i in range(0, len(samples), 500):              # 500 cells at a time keeps memory low
            batch = pf.read(columns=samples[i:i + 500]).to_pandas()
            blocks.append(sp.csr_matrix(batch.to_numpy(dtype=np.int32).T))
        X = sp.vstack(blocks).tocsr()                       # cells x genes
        samples = pd.Index(samples)
    Log.check(f"mouse counts: {X.shape[1]} genes x {X.shape[0]} samples, {X.nnz/1e6:.1f} M non-zeros")

    # samples -> cells (transcriptomics_sample_id -> cell_id)
    sid_to_cell = dict(zip(cells["transcriptomic_sample_id"], cells["cell_id"]))
    keep = np.array([s in sid_to_cell for s in samples])
    Log.check(f"mouse counts: {keep.sum()}/{len(samples)} samples matched to a cell")
    X = X[np.flatnonzero(keep)]
    obs_names = [sid_to_cell[s] for s in np.asarray(samples)[keep]]

    # library size = total counts of each cell (used for CPM normalization)
    lib = np.asarray(X.sum(axis=1)).ravel().astype(np.float64)
    if (lib == 0).any():
        Log.warn(f"mouse: {(lib == 0).sum()} cells with zero counts")
    a = ad.AnnData(X=X,
                   obs=pd.DataFrame({"library_size": lib}, index=pd.Index(obs_names, name="cell_id")),
                   var=pd.DataFrame(index=pd.Index(genes, name="gene")))
    a.var_names_make_unique()
    a.uns["description"] = ("Gouwens et al. 2020 mouse Patch-seq. X = raw read counts (int32). "
                            "CPM = X / obs['library_size'] * 1e6; the course uses log2(CPM + 1).")
    a.uns["values"] = "counts"
    return a


def log2cpm(counts: np.ndarray, library_size: np.ndarray) -> np.ndarray:
    """log2(counts / library_size * 1e6 + 1), the normalization used throughout the course."""
    return np.log2(counts / library_size[:, None] * 1e6 + 1)

## 7. Human (Lee, Dalley et al. 2023)

All human inputs come from the authors' [GitHub repository](https://github.com/AllenInstitute/human_patchseq_gaba), pinned to one commit.

In [ ]:
def build_human(args) -> dict:
    Log.info("\n=== HUMAN (Lee, Dalley et al. 2023) ===")
    import rdata
    cache = args.cache
    meta = pd.read_csv(fetch("human_metadata", cache, args.offline))
    meta.columns = meta.columns.str.strip()          # 'Transcriptomic_type ' has a trailing space
    dup = meta["specimen_id_x"].duplicated(keep="first")
    if dup.any():
        Log.warn(f"human metadata: dropping {dup.sum()} duplicated specimen row(s): "
                 f"{meta.loc[dup, 'specimen_id_x'].tolist()}")
        meta = meta[~dup]
    Log.check(f"human metadata: {len(meta)} cells")

    # annotation table inside the RData file has sex and depth
    r2 = rdata.read_rda(str(fetch("human_expr_2", cache, args.offline)))
    anno = r2["annoPatch"].set_index("cell_name_label")

    c = pd.DataFrame({
        "cell_id": "human_" + meta["specimen_id_x"].astype(str),
        "species": "human",
        "dataset": "LeeDalley2023",
        "specimen_id": meta["specimen_id_x"].astype("Int64"),
        "cell_name": meta["cell_name"],
        "donor_id": meta["Donor"],
        "sex": meta["cell_name"].map(anno["gender_label"]),
        "structure": meta["structure"],
        "area": meta["structure"],
        "lobe": meta["lobe"],
        "layer": meta["Cortical_layer"].astype("string").str.replace(r"\.0$", "", regex=True),
        # 0 is used for 'not measured' in the source table, so it becomes missing here
        "depth_from_pia_um": pd.to_numeric(meta["cell_name"].map(anno["Depth_from_Pia_microns_label"]),
                                           errors="coerce").replace(0, np.nan),
        "disease": meta["disease_category"],
        "preparation": meta["condition"],                 # acute slice or slice culture
        "days_in_culture": meta["Days_In_culture"],
        "virus": meta["Virus"],
        "t_type": meta["Transcriptomic_type"],
        "subclass": meta["Revised_subclass_label"],
        "class": meta["broad_class_label"],
        "cross_species_type": meta["M_H_Hodge_cross_species_types"],
        "morphology_type_qualitative": meta["Qualitative_morphology_type"],
        "transcriptomic_sample_id": meta["patched_cell_container"],
        "ephys_session_id": to_int_or_na(meta["nwb_id"]),
        "has_ephys": meta["Has_ephys"].eq(True),
        "has_morphology": meta["Has_morphology"].eq(True),
        "in_published_analysis": True,
        "nwb_url": pd.NA,
        "swc_url": pd.NA,
    })

    out = {}
    for kind, key in [("ephys", "human_ephys"), ("morph", "human_morph")]:
        t = pd.read_csv(fetch(key, cache, args.offline))
        t = t.drop(columns=[x for x in t.columns if x.startswith("Unnamed")])
        rename = {o: snake(o) for o in t.columns if o != "specimen_id"}
        t = t.rename(columns=rename)
        t.insert(0, "cell_id", "human_" + t["specimen_id"].astype(str))
        miss = ~t["cell_id"].isin(c["cell_id"])
        Log.check(f"human {kind} features: {len(t)} cells x {len(rename)} features; "
                  f"{miss.sum()} not in metadata")
        out[f"human_{kind}_features"] = (t, rename, key)
    # flags must agree with the feature tables
    c["has_ephys"] |= c["cell_id"].isin(out["human_ephys_features"][0]["cell_id"])
    c["has_morphology"] |= c["cell_id"].isin(out["human_morph_features"][0]["cell_id"])

    if not args.skip_expression:
        out["human_expression"] = human_expression(args, c, r2)
        c["has_transcriptome"] = c["cell_id"].isin(out["human_expression"].obs_names)
    else:
        c["has_transcriptome"] = c["transcriptomic_sample_id"].notna()

    c = attach_dandi(c, "human", args.skip_dandi)
    out["cells"] = c
    return out


def human_expression(args, cells: pd.DataFrame, r2: dict):
    import anndata as ad
    import rdata
    d1 = rdata.read_rda(str(fetch("human_expr_1", args.cache, args.offline)))["datPatch1"]
    d2 = r2["datPatch2"]
    genes = d1.coords["dim_0"].values.astype(str)
    assert (genes == d2.coords["dim_0"].values.astype(str)).all(), "gene order differs between parts"
    samples = np.concatenate([d1.coords["dim_1"].values, d2.coords["dim_1"].values]).astype(str)
    X = sp.hstack([sp.csc_matrix(d1.values), sp.csc_matrix(d2.values)]).T.tocsr()   # cells x genes
    sums = np.asarray(_unlog(X).sum(axis=1)).ravel()     # undo log2(x+1): should give CPM sums of 1e6
    Log.check(f"human expression: {X.shape[1]} genes x {X.shape[0]} samples; values are log2(CPM+1) "
              f"(CPM sums per cell: min {sums.min():.0f}, max {sums.max():.0f}; expected 1e6)")
    sid_to_cell = dict(zip(cells["transcriptomic_sample_id"], cells["cell_id"]))
    keep = np.array([s in sid_to_cell for s in samples])
    Log.check(f"human expression: {keep.sum()}/{len(samples)} samples matched to a cell")
    X = X[np.flatnonzero(keep)]
    a = ad.AnnData(X=X.astype(np.float32),
                   obs=pd.DataFrame(index=pd.Index([sid_to_cell[s] for s in samples[keep]], name="cell_id")),
                   var=pd.DataFrame(index=pd.Index(genes, name="gene")))
    a.var_names_make_unique()
    a.uns["description"] = ("Lee, Dalley et al. 2023 human Patch-seq. X = log2(CPM+1) as published "
                            "(raw counts are not part of this release).")
    a.uns["values"] = "log2cpm"
    return a


def _unlog(X: sp.csr_matrix) -> sp.csr_matrix:
    Y = X.copy()
    Y.data = np.power(2.0, Y.data) - 1
    return Y

## 8. Gene selection, data dictionary, writing

In [ ]:
def selected_genes(args, mouse_genes, human_genes) -> pd.DataFrame:
    vg = pd.read_csv(fetch("vgic_genes", args.cache, args.offline))
    rows = []
    for _, r in vg.iterrows():
        fam = re.sub(r"<[^>]+>", "", str(r["Family name"])).strip()
        rows.append({"human_symbol": r["HGNC symbol"], "mouse_symbol": human_symbol_to_mouse(r["HGNC symbol"]),
                     "group": "voltage-gated ion channel", "family": fam, "source": "vgic_genes.csv (Allen)"})
    for g in MARKER_GENES:
        rows.append({"human_symbol": MOUSE_TO_HUMAN_EXCEPTIONS.get(g, g.upper()), "mouse_symbol": g, "group": "marker",
                     "family": "", "source": "course marker list"})
    t = pd.DataFrame(rows).drop_duplicates("mouse_symbol")
    t["in_mouse_data"] = t["mouse_symbol"].isin(set(mouse_genes)) if mouse_genes is not None else pd.NA
    t["in_human_data"] = t["human_symbol"].isin(set(human_genes)) if human_genes is not None else pd.NA
    for sp_name, col in [("mouse", "in_mouse_data"), ("human", "in_human_data")]:
        if t[col].notna().all():
            miss = t.loc[~t[col].astype(bool), f"{sp_name}_symbol"].tolist()
            Log.check(f"selected genes: {int(t[col].sum())}/{len(t)} found in {sp_name} data"
                      + (f"; missing: {', '.join(miss)}" if miss else ""))
    return t


def write_selected(a, genes: list[str], path: Path) -> None:
    genes = [g for g in genes if g in a.var_names]
    sub = a[:, genes].X
    sub = sub.toarray() if sp.issparse(sub) else np.asarray(sub)
    if a.uns["values"] == "counts":
        sub = log2cpm(sub.astype(np.float64), a.obs["library_size"].to_numpy())
    pd.DataFrame(sub.astype(np.float32), index=a.obs_names, columns=genes).to_parquet(path)


CELL_COLUMNS = {
    "cell_id": "Course-wide unique id: '<species>_<specimen_id>'. Use this to join all tables.",
    "species": "mouse or human",
    "dataset": "Gouwens2020 (mouse V1) or LeeDalley2023 (human neocortex)",
    "specimen_id": "Allen cell specimen id (the id used on brain-map.org)",
    "cell_name": "Allen cell name (human only)",
    "donor_id": "Animal / patient donor id",
    "sex": "Biological sex of the donor",
    "age_days": "Postnatal age in days (mouse only)",
    "genotype": "Transgenic line used to target the cell (mouse only)",
    "hemisphere": "Brain hemisphere (mouse only)",
    "structure": "Allen structure acronym as published (mouse: area + layer, e.g. VISp5)",
    "area": "Cortical area (mouse: VISp, VISl, ...; human: MTG, SFG, ...)",
    "lobe": "Cortical lobe (human only)",
    "layer": "Cortical layer of the soma",
    "soma_depth_normalized": "Soma depth, 0 = pia, 1 = white matter (mouse only)",
    "depth_from_pia_um": "Soma depth from pia in micrometers (human only)",
    "dendrite_type": "aspiny / sparsely spiny / spiny (mouse only)",
    "reconstruction_type": "full / dendrite-only morphology reconstruction (mouse only)",
    "disease": "Reason for neurosurgery (human only): epilepsy, tumor, ...",
    "preparation": "acute slice, or slice culture (human: 'culture' cells were kept in culture and often virally labeled)",
    "days_in_culture": "Days in slice culture (human only)",
    "virus": "Viral label used in culture (human only)",
    "t_type": "Transcriptomic type (mouse: Tasic 2018 taxonomy; human: Lee/Dalley 2023 types)",
    "t_type_probability": "Confidence of the t-type assignment (mouse only)",
    "subclass": "Transcriptomic subclass as published (mouse: Pvalb, Sst, Sst Chodl, Vip, Lamp5, Sncg; human: PVALB, SST, VIP, LAMP5/PAX6/Other)",
    "subclass_harmonized": "Coarse cross-species subclass made by this script (see SUBCLASS_HARMONIZED in 00_Build_Course_Data.ipynb). An approximation for comparisons, not a published homology.",
    "class": "Broader class (mouse: MGE / CGE; human: GABAergic neuron)",
    "met_type": "Morpho-electric-transcriptomic type, Gouwens 2020 (mouse only, ~490 cells)",
    "cross_species_type": "Mouse-human homologous type for a subset of human cells (Hodge et al. 2019 labels)",
    "morphology_type_qualitative": "Expert morphology call (human only)",
    "transcriptomic_sample_id": "Id of the sequencing sample; links the cell to its expression column",
    "ephys_session_id": "Id of the electrophysiology session; appears as 'ses-<id>' in the DANDI file name",
    "in_published_analysis": "True if the cell is part of the published analysis (mouse: present in the Brain Knowledge Platform specimen table)",
    "has_ephys": "Electrophysiology available",
    "has_morphology": "Morphology reconstruction available",
    "has_transcriptome": "Cell has a column in the expression matrix",
    "nwb_url": "Direct download URL of the raw NWB file (DANDI asset). Stream it with pynwb + remfile.",
    "swc_url": "Download URL of the reconstruction (SWC, Brain Image Library; mouse only)",
    "dandiset": "DANDI dataset id",
    "dandi_asset_id": "DANDI asset id of the NWB file",
    "dandi_path": "Path of the NWB file inside the dandiset",
    "dandi_version": "Dandiset version that was queried",
}


def write_dictionary(path: Path, tables: dict, cells: pd.DataFrame) -> None:
    L = ["# Course data dictionary", "",
         f"Generated by `00_Build_Course_Data.ipynb` v{SCRIPT_VERSION} on {dt.date.today().isoformat()}.",
         "All tables join on `cell_id`. Sources and checksums are in `provenance.json`.", "",
         "## cells.csv", "", f"{len(cells)} rows "
         f"({(cells.species == 'mouse').sum()} mouse, {(cells.species == 'human').sum()} human).", "",
         "| column | non-null | description |", "|---|---|---|"]
    for col in cells.columns:
        L.append(f"| `{col}` | {cells[col].notna().sum()} | {CELL_COLUMNS.get(col, '')} |")
    for name, (t, rename, src) in tables.items():
        L += ["", f"## {name}.csv", "",
              f"{len(t)} cells. Source: `{src}` ({SOURCES[src]['about']})", "",
              "| column | original column name | non-null |", "|---|---|---|"]
        inv = {v: k for k, v in rename.items()}
        for col in t.columns:
            L.append(f"| `{col}` | {inv.get(col, '')} | {t[col].notna().sum()} |")
    L += ["", "## Notes on units", "",
          "* Mouse feature names carry units from the Allen table: `_mv` millivolts, `_pa` picoamps, "
          "`_mm` millimeters. `tau`, `latency`, `avg_isi` are as in the Allen Cell Types documentation.",
          "* Human features use IPFX names; units follow IPFX conventions and have NOT been converted. "
          "Mouse and human features come from different pipeline versions: check definitions before "
          "comparing species, or recompute features from the NWB files with one pipeline.",
          "", "## Expression files", "",
          "* `mouse_expression.h5ad`: AnnData, cells x genes, sparse. `X` = raw read counts; "
          "`obs['library_size']` = total counts per cell, so CPM = X / library_size * 1e6.",
          "* `human_expression.h5ad`: AnnData, cells x genes, sparse. `X` = log2(CPM+1) as published "
          "(the human release has no raw counts).",
          "* Open either with `anndata.read_h5ad(path)` (or scanpy); `uns['values']` says which it is.",
          "* `*_expression_selected.parquet`: log2(CPM+1) of the genes in `genes_selected.csv` as a plain "
          "pandas table (`pd.read_parquet`), index = `cell_id`. Same normalization for both species.",
          "* Mouse CPM uses the total counts of each cell in the NeMO `count.v2` matrix. "
          "Human CPM is the published value. Both are log2(CPM+1), but they come from different "
          "pipelines: compare *patterns* across species, not absolute values.",
          ]
    path.write_text("\n".join(L) + "\n")


def package_versions() -> dict:
    out = {"python": platform.python_version()}
    for p in ["pandas", "numpy", "scipy", "pyarrow", "anndata", "rdata", "openpyxl", "requests"]:
        try:
            out[p] = importlib_metadata.version(p)
        except importlib_metadata.PackageNotFoundError:
            out[p] = None
    return out

## 9. Run the build

The cells above only defined things. From here on the data is downloaded and processed.

In [ ]:
args.out.mkdir(parents=True, exist_ok=True)
args.cache.mkdir(parents=True, exist_ok=True)
Log.lines.clear()                        # so that re-running the notebook starts a fresh report
PROVENANCE.clear()
PROVENANCE.update({"sources": {}, "dandi": {}})
t0 = time.time()

results = {}
if args.only in (None, "mouse"):
    results["mouse"] = build_mouse(args)

In [ ]:
if args.only in (None, "human"):
    results["human"] = build_human(args)

In [ ]:
Log.info("\n=== WRITING ===")
cells = pd.concat([r["cells"] for r in results.values()], ignore_index=True)
cells["subclass_harmonized"] = cells["subclass"].map(SUBCLASS_HARMONIZED)
order = [k for k in CELL_COLUMNS if k in cells.columns]
cells = cells[order + [k for k in cells.columns if k not in order]]
assert cells["cell_id"].is_unique, "cell_id must be unique"
cells.to_csv(args.out / "cells.csv", index=False)

tables = {}
for r in results.values():
    for k, v in r.items():
        if k.endswith("_features"):
            tables[k] = v
            v[0].to_csv(args.out / f"{k}.csv", index=False)

mouse_a = results.get("mouse", {}).get("mouse_expression")
human_a = results.get("human", {}).get("human_expression")
if mouse_a is not None or human_a is not None:
    g = selected_genes(args, None if mouse_a is None else mouse_a.var_names,
                       None if human_a is None else human_a.var_names)
    g.to_csv(args.out / "genes_selected.csv", index=False)
    if mouse_a is not None:
        write_selected(mouse_a, g["mouse_symbol"].tolist(), args.out / "mouse_expression_selected.parquet")
        mouse_a.write_h5ad(args.out / "mouse_expression.h5ad", compression="gzip")
    if human_a is not None:
        write_selected(human_a, g["human_symbol"].tolist(), args.out / "human_expression_selected.parquet")
        human_a.write_h5ad(args.out / "human_expression.h5ad", compression="gzip")

# summary
Log.info("")
for spc, grp in cells.groupby("species"):
    Log.check(f"SUMMARY {spc}: {len(grp)} cells | ephys {grp.has_ephys.sum()} | morphology "
              f"{grp.has_morphology.sum()} | transcriptome {grp.has_transcriptome.sum()} | all three "
              f"{(grp.has_ephys & grp.has_morphology & grp.has_transcriptome).sum()} | NWB link "
              f"{grp.nwb_url.notna().sum()}")
    Log.check(f"SUMMARY {spc} subclasses: {grp.subclass.value_counts().to_dict()}")

write_dictionary(args.out / "DATA_DICTIONARY.md", tables, cells)
(args.out / "validation_report.txt").write_text("\n".join(Log.lines) + "\n")
PROVENANCE.update({
    "script": "00_Build_Course_Data.ipynb", "script_version": SCRIPT_VERSION,
    "built": dt.datetime.now().isoformat(timespec="seconds"),
    "arguments": {k: str(v) for k, v in vars(args).items()},
    "environment": package_versions(),
    "outputs": {p.name: {"sha256": sha256(p), "size_bytes": p.stat().st_size}
                for p in sorted(args.out.iterdir()) if p.is_file() and p.name != "provenance.json"},
})
(args.out / "provenance.json").write_text(json.dumps(PROVENANCE, indent=2, default=str))
Log.info(f"\nDone in {time.time() - t0:.0f} s. Output: {args.out.resolve()}")

## 10. Check the result

Read the validation report: it lists every warning and the counts and join rates the build achieved.
`DATA_DICTIONARY.md` in `data/` describes every column.

In [ ]:
print((args.out / "validation_report.txt").read_text())

In [ ]:
cells = pd.read_csv(args.out / "cells.csv")
cells.groupby(["species", "subclass_harmonized"]).size().unstack(0)